# Preprocessing for Clustering

This notebook creates the first model-ready preprocessing output from the customer-level feature table. It does not run clustering or modelling.

## Imports

Use the reusable preprocessing functions from `src/preprocessing.py`.

In [1]:
import sys

import pandas as pd

sys.path.append("../src")

from preprocessing import clean_feature_values, preprocess_for_clustering, select_model_features

## Load Customer Features

Load the combined customer-level feature table created during feature engineering.

In [2]:
customer_features = pd.read_csv("../data/processed/customer_features_info.csv")
customer_features.head()


,customer_id,customer_gender,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_total_distinct_products,year_first_transaction,percentage_of_products_bought_promotion,typical_hour,...,share_electronics,share_vegetables,share_nonalcohol_drinks,share_alcohol_drinks,share_meat,share_fish,share_hygiene,share_videogames,share_petfood,degree_level
0,3,female,1.0,1.0,1.0,3.0,189.0,2020.0,0.631599,NaN,...,0.244917,0.020065,0.017375,0.009521,0.001506,0.011458,0.029693,0.013771,0.020656,Bsc
1,4,female,1.0,0.0,0.0,2.0,130.0,2013.0,0.149890,NaN,...,0.047596,0.099442,0.026343,0.004695,0.002125,0.000741,0.092918,0.016458,0.032867,Bsc
2,5,male,0.0,0.0,NaN,2.0,81.0,2005.0,0.069126,11.0,...,0.000000,0.035694,0.006496,0.007589,0.081356,0.017557,0.032607,0.006496,0.014277,Msc
3,7,male,0.0,0.0,2.0,1.0,92.0,2021.0,0.253609,18.0,...,0.073903,0.005618,0.050629,0.075776,0.065008,0.072432,0.032437,0.110754,0.012306,Unknown
4,8,male,0.0,0.0,3.0,1.0,6.0,2021.0,0.186569,17.0,...,0.420243,0.014730,0.022948,0.027833,0.041400,0.039346,0.011513,0.048765,0.017095,Unknown


## Initial Checks

Check shape, duplicate customer IDs, and missing values before preprocessing.

In [3]:
print(f"Rows before preprocessing: {customer_features.shape[0]:,}")
print(f"Columns before preprocessing: {customer_features.shape[1]:,}")
print(f"Duplicated customer_id before preprocessing: {customer_features['customer_id'].duplicated().sum():,}")

Rows before preprocessing: 33,038
Columns before preprocessing: 29
Duplicated customer_id before preprocessing: 0


In [4]:
missing_before = customer_features.isna().sum().sort_values(ascending=False)
missing_before[missing_before > 0]

share_fish                                 991
typical_hour                               661
share_petfood                              661
share_meat                                 661
share_electronics                          661
share_videogames                           661
share_vegetables                           661
number_complaints                          661
total_children_home                        656
distinct_stores_visited                    330
percentage_of_products_bought_promotion    330
share_hygiene                              330
teens_home                                 330
kids_home                                  330
share_alcohol_drinks                       330
age                                        165
dtype: int64

## Clean and Select Features

Clean invalid and missing values, then remove non-model columns while keeping `customer_id` for traceability.

In [5]:
cleaned_customer_features = clean_feature_values(customer_features)
selected_customer_features = select_model_features(cleaned_customer_features)

removed_columns = [
    column for column in customer_features.columns
    if column not in selected_customer_features.columns
]

removed_columns


[]

In [6]:
missing_after_cleaning = selected_customer_features.isna().sum().sort_values(ascending=False)
missing_after_cleaning[missing_after_cleaning > 0]

Series([], dtype: int64)

## Preprocess Model Features

One-hot encode categorical model columns and scale numeric model features with `StandardScaler`. `customer_id` is preserved and not used as a model feature.

In [7]:
customer_features_model = preprocess_for_clustering(customer_features)

selected_feature_columns = [
    "customer_id",
    "number_complaints",
    "distinct_stores_visited",
    "lifetime_total_distinct_products",
    "percentage_of_products_bought_promotion",
    "typical_hour",
    "age",
    "customer_tenure",
    "has_loyalty_card",
    "total_children_home",
    "total_lifetime_spend",
    "share_groceries",
    "share_electronics",
    "share_vegetables",
    "share_nonalcohol_drinks",
    "share_alcohol_drinks",
    "share_meat",
    "share_fish",
    "share_hygiene",
    "share_videogames",
    "share_petfood",
    "customer_gender_female",
    "degree_level_Bsc",
    "degree_level_Msc",
    "degree_level_Phd",
    "degree_level_Unknown",
]

selected_model_features = customer_features_model[selected_feature_columns].copy()
selected_model_features.head()


,customer_id,number_complaints,distinct_stores_visited,lifetime_total_distinct_products,percentage_of_products_bought_promotion,typical_hour,age,customer_tenure,has_loyalty_card,total_children_home,...,share_meat,share_fish,share_hygiene,share_videogames,share_petfood,customer_gender_female,degree_level_Bsc,degree_level_Msc,degree_level_Phd,degree_level_Unknown
0,3,0.076516,-0.099810,0.378445,1.131040,-0.134436,0.048856,-0.931644,0.810886,-0.008470,...,-1.143197,-0.691773,-0.327461,-0.186336,0.163844,1,1,0,0,0
1,4,-1.052532,-0.700126,-0.178573,-0.643434,-0.134436,-0.228627,0.459419,0.810886,-0.574949,...,-1.122408,-1.143449,1.227758,-0.073252,1.025451,1,1,0,0,0
2,5,0.076516,-0.700126,-0.641180,-0.940948,-0.342479,-0.006640,2.049206,-1.233219,-1.141429,...,1.538352,-0.434685,-0.255800,-0.492471,-0.286251,0,0,1,0,0
3,7,1.205564,-1.300442,-0.537330,-0.261363,1.113819,-0.617103,-1.130368,0.810886,-1.141429,...,0.989355,1.878166,-0.259969,3.894636,-0.425354,0,0,0,0,1
4,8,2.334612,-1.300442,-1.349253,-0.508321,0.905776,0.104353,-1.130368,0.810886,-1.141429,...,0.196543,0.483649,-0.774672,1.286198,-0.087441,0,0,0,0,1


## Final Validation

Confirm the preprocessed output keeps all customers, has no duplicate IDs, and has no missing values in model features.

In [8]:
preprocessing_validation = pd.DataFrame(
    {
        "check": [
            "rows before preprocessing",
            "rows after preprocessing",
            "columns before preprocessing",
            "selected columns including customer_id",
            "duplicated customer_id after preprocessing",
            "missing values in selected model features",
        ],
        "value": [
            customer_features.shape[0],
            selected_model_features.shape[0],
            customer_features.shape[1],
            selected_model_features.shape[1],
            selected_model_features["customer_id"].duplicated().sum(),
            selected_model_features.drop(columns=["customer_id"]).isna().sum().sum(),
        ],
    }
)

preprocessing_validation


,check,value
0,rows before preprocessing,33038
1,rows after preprocessing,33038
2,columns before preprocessing,29
3,selected columns including customer_id,26
4,duplicated customer_id after preprocessing,0
5,missing values in selected model features,0


In [9]:
missing_after = selected_model_features.drop(columns=["customer_id"]).isna().sum().sort_values(ascending=False)
missing_after[missing_after > 0]


Series([], dtype: int64)

In [10]:
selected_model_features.columns.to_frame(index=False, name="feature_column")


,feature_column
0,customer_id
1,number_complaints
2,distinct_stores_visited
3,lifetime_total_distinct_products
4,percentage_of_products_bought_promotion
5,typical_hour
6,age
7,customer_tenure
8,has_loyalty_card
9,total_children_home


## Save Preprocessed Feature Table

Save the preprocessed customer-level feature table. This file keeps `customer_id` for traceability, but clustering should use the remaining feature columns.

In [11]:
selected_model_features.to_csv("../data/processed/selected_model_features.csv", index=False)
print("Saved ../data/processed/selected_model_features.csv")


Saved ../data/processed/selected_model_features.csv


## Preprocessing Notes

- `customer_id` is kept in the final file for traceability, but it is excluded from model feature scaling.
- `most_frequent_product` is removed because it is useful for profiling but not a simple numeric clustering feature.
- Numeric missing values are filled with the median, and categorical missing values are filled with `Unknown`.
- Invalid `percentage_of_products_bought_promotion` values are clipped to the `0` to `100` range.
- `customer_gender` and `degree_level` are one-hot encoded.
- Numeric model features are scaled with `StandardScaler`.
- No clustering or modelling is performed in this notebook.